## Import

In [ ]:
import pandas as pd
import json

import re
from difflib import get_close_matches
import hashlib

import importlib
import pipeline.utils

importlib.reload(pipeline.utils)

from pipeline.utils import score_data_quality, summarize_quality

In [127]:
df = pd.read_csv('https://raw.githubusercontent.com/leonardo-alexander/titanic-dataset/refs/heads/main/train.csv')

## Standardize Schema

In [128]:
def normalize(col):
    return re.sub(r'[^a-z]', '', col.lower())

### Gender Column Naming Standard

In [129]:
GENDER_KEYWORDS = [
    "gender",
    "sex",
    "gndr"
]

In [130]:
def match_gender(col):
    norm = normalize(col)

    # 1. Strong direct match
    for keyword in GENDER_KEYWORDS:
        if norm == keyword:
            return "Gender"

    # 2. Containment match (safe)
    for keyword in GENDER_KEYWORDS:
        if keyword in norm and len(norm) <= len(keyword) + 6:
            return "Gender"

    # 3. Fuzzy match (for typos)
    match = get_close_matches(norm, GENDER_KEYWORDS, n=1, cutoff=0.8)
    if match:
        return "Gender"

    return None

In [131]:
def standardize_column(col):
    gender = match_gender(col)
    if gender:
        return gender

    return col

df.columns = [standardize_column(col) for col in df.columns]

### PascalCase

In [132]:
def to_pascal_case(col):
    words = re.split(r'[_\s]+', col)
    return ''.join(word.capitalize() for word in words if word)

df.columns = [to_pascal_case(col) for col in df.columns]

## Remove Sensitive Data (PII)

In [133]:
def hash_value(val):
    return hashlib.sha256(str(val).encode()).hexdigest()

In [134]:
def detect_pii_action(col):
    norm = normalize(col)

    keyword_rules = {
        "name": "drop",
        "email": "drop",
        "phone": "drop",
        "address": "drop",
        "id": "hash"
    }

    NAME_EXCEPTIONS = {"CompanyName", "ProductName"}

    for key, action in keyword_rules.items():

        # Safer ID detection
        if key == "id":
            if norm.endswith("id"):
                return action

        # Safer Name detection
        elif key == "name":
            if key in norm and col not in NAME_EXCEPTIONS:
                return action

        # Default rule
        elif key in norm:
            return action

    return None

In [135]:
def handle_pii(df):
    df = df.copy()
    pii_report = []

    for col in df.columns:
        action = detect_pii_action(col)

        if action == "drop":
            df.drop(columns=[col], inplace=True)
            pii_report.append({
                "column": col,
                "action": "dropped"
            })

        elif action == "hash":
            df[col] = df[col].apply(hash_value)
            pii_report.append({
                "column": col,
                "action": "hashed"
            })

    return df, pii_report

## Summary

In [136]:
def run_step1(df):
    reports = {}

    # PII
    df, pii_report = handle_pii(df)
    reports["pii"] = pii_report

    # Quality scoring
    df = score_data_quality(df)
    quality_summary = summarize_quality(df)

    reports["quality"] = quality_summary

    return df, reports

In [137]:
df, reports = run_step1(df)

## Export

In [ ]:
df.to_csv('data/minimum/minimum_cleaning.csv.zip', index=False, compression='zip')

with open('data/minimum/minimum_cleaning_report.json', 'w') as f:
    json.dump(reports, f, indent=2)